In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
#Will be used for smoothing
from scipy.signal import savgol_filter, find_peaks

In [ ]:
#Obtaining metadata for the video to calculate duration
cap = cv2.VideoCapture('Videos/vidssave.com How 5K running paces looks on a treadmill! 15 minutes 5K. 720P.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
duration = frame_count / fps
cap.release()

In [ ]:
model = YOLO("yolo26n-pose.pt")  # load a pretrained model (recommended for training)
#stream=True now allows us to process the video frame by frame
source = model(source='/Users/abhinavarora/Desktop/CadenceCV/Videos/vidssave.com How 5K running paces looks on a treadmill! 15 minutes 5K. 720P.mp4', show=False, conf=0.3, stream=True)
#Result for each frame
right_ankle_coords = []
left_ankle_coords = []
right_hip_coords = []
left_hip_coords = []
left_knee_coords = []
right_knee_cords = []
left_shoulder_coords = []
right_shoulder_coords = []
frames = []
#Frame index
i = 0
for result in source:
    #Obtaining the first person's keypoints from the video
    if len(result.keypoints) == 0:
        continue
    #Extracting essential keypoints based off indices according to the YOLOv8 documentation
    kpts = result.keypoints.xy[0]
    left_ankle = kpts[15]
    right_ankle = kpts[16]
    left_hip = kpts[11]
    right_hip = kpts[12]
    left_knee = kpts[13]
    right_knee = kpts[14]
    right_shoulder = kpts[6]
    left_shoulder = kpts[5]
    right_ankle_coords.append(right_ankle)
    left_ankle_coords.append(left_ankle)
    right_hip_coords.append(right_hip)
    left_hip_coords.append(left_hip)
    left_knee_coords.append(left_knee)
    right_knee_cords.append(right_knee)
    left_shoulder_coords.append(left_shoulder)
    right_shoulder_coords.append(right_shoulder)

    i += 1
    frames.append(i)

frames = np.array(frames)
right_ankle_coords = np.array(right_ankle_coords)
left_ankle_coords = np.array(left_ankle_coords)
right_hip_coords = np.array(right_hip_coords)
left_hip_coords = np.array(left_hip_coords)
left_knee_coords = np.array(left_knee_coords)
right_knee_cords = np.array(right_knee_cords)


In [ ]:
#Angle metrics
left_knee_flexion_angle = []
right_knee_flexion_angle = []

In [ ]:
#Returns vertical oscillations and cadence
def metrics():
    #Applying the savgol filter to smoothen it out and remove noise -> Easier for peak detection for cadence
    #find_peaks functions returns a tuple -> (arr, dict). dict must be discarded it isn't useful
    window_length = 9
    poly_order = 2
    right_ankle_coords_smooth = savgol_filter(right_ankle_coords[:,1], window_length=window_length, polyorder=poly_order)
    right_ankle_peaks, _ = find_peaks(right_ankle_coords_smooth, None, None, 8)

    left_ankle_coords_smooth = savgol_filter(left_ankle_coords[:,1], window_length=window_length, polyorder=poly_order)
    left_ankle_peaks, _ = find_peaks(left_ankle_coords_smooth, None, None, 8)

    #The idea is that each peak corresponds to a step since when stepping the foot has to come to the highest y-axis. So this applies for both 
    #feet so they're summed up. The total is averaged over the video duration
    cadence = (len(right_ankle_peaks) + len(left_ankle_peaks)) / duration

    right_hip_coords_smooth = savgol_filter(right_hip_coords[:,1], window_length=window_length, polyorder=poly_order)

    left_hip_coords_smooth = savgol_filter(left_hip_coords[:,1], window_length=window_length, polyorder=poly_order)

    #Finding the vertical oscillation is essentially checking the average oscillation during each stride. Strides can be determined via peaks
    #on the ankle position graphs -> Higher the peak number, lower it is on the screen
    #Finding the correct slice depending upon peaks in the ankle positioning
    def calculate_avg_hip_oscillations(peak_arr, smooth_hip_coords):
        #sum of hip_oscillations
        s = 0
        for i, peak in enumerate(peak_arr):
            if i + 1 < len(peak_arr):
                hip_frame_slice = smooth_hip_coords[peak: peak_arr[i+1] + 1]
                max_pos = max(hip_frame_slice)
                min_pos = min(hip_frame_slice)
                res = max_pos - min_pos
                s += res
        
        return s/len(peak_arr)
    
    #Calculates knee flexion angle, tibial angles, ankle-hip horizontal offset, 
    #trunk_angle, pre-strike-ankle-velocity, ankle_movement direction pre strikefoot
    #This will be used to build a dataset for detecting
    #All these are combined since they are measured at strikefoot
    def calculate_angles_and_metrics(peak_arr, knee_coords, hip_coords, ankle_coords, shoulder_coords, ankle_coords_smoothed):
        knee_flexion_angles = []
        tibial_angles = []
        ankle_hip_horizontal_offset = []
        trunk_angles = []
        ankle_velocities = []
        ankle_approach_angles = []
        all_ankle_velocities = np.gradient(ankle_coords_smoothed)
        for index in peak_arr:
            if index < len(knee_coords) and index < len(hip_coords) and index < len(shoulder_coords):
                #Knee flexion
                vector_1 = np.array(hip_coords[index]) - np.array(knee_coords[index])
                vector_2 = np.array(ankle_coords[index]) - np.array(knee_coords[index])
                #np.arctan2 gives angles relative to the horizontal, so the angles must be subtracted
                angle1 = np.arctan2(vector_1[1], vector_1[0])
                angle2 = np.arctan2(vector_2[1], vector_2[0])
                final_angle = np.rad2deg(angle2 - angle1) % 360
                knee_flexion_angles.append(final_angle)

                #Tibial angle: -> angle between the knee and the ankle  w.r.t the vertical and not the horizontal
                angle1 = np.arctan2(vector_2[1], vector_2[0])
                final_angle = np.pi/2 - angle1
                tibial_angles.append(final_angle)

                #Ankle-hip horizontal offset
                offset = np.abs(ankle_coords[index][0] - hip_coords[index][0])
                ankle_hip_horizontal_offset.append(offset)

                #Trunk angle : -> angle between the shoulder and the hip w.r.t the vertical
                vector_1 = np.array(shoulder_coords[index]) - np.array(hip_coords[index])
                angle1 = np.pi/2 - np.arctan2(vector_1[1], vector_1[0])
                final_angle = np.rad2deg(angle1)
                trunk_angles.append(final_angle)

                #Pre-strikefoot ankle velocity (2 frames before strike foot)
                if index - 3 > -1:
                    vel = all_ankle_velocities[index - 2]
                    ankle_velocities.append(vel)
                
                    #Pre-strikefoot ankle movement direction angle
                    dx = ankle_coords[index - 1, 0] - ankle_coords[index - 3, 0]
                    dy = ankle_coords[index - 1, 1] - ankle_coords[index - 3, 1]
                    res = np.arctan2(dy, dx)
                    ankle_approach_angles.append(res)

        
        return {"Knee Flexion angle": knee_flexion_angles, 
                "Tibial angles": tibial_angles, 
                "Ankle Hip Horizontal Offset": ankle_hip_horizontal_offset, 
                "Trunk Angles": trunk_angles, 
                "Ankle Velocities": ankle_velocities,
                "Ankle Approach angles": ankle_approach_angles}

    left_side_metrics = calculate_angles_and_metrics(left_ankle_peaks, left_knee_coords, left_hip_coords, left_ankle_coords, left_shoulder_coords, left_ankle_coords_smooth)
    right_side_metrics = calculate_angles_and_metrics(right_ankle_peaks, right_knee_cords, right_hip_coords, right_ankle_coords, right_shoulder_coords, right_ankle_coords_smooth)
    left_knee_flexion, left_tibial_angles, left_ankle_hip_offset = left_side_metrics["Knee Flexion angle"], left_side_metrics["Tibial angles"], left_side_metrics["Ankle Hip Horizontal Offset"]
    right_knee_flexion, right_tibial_angles, right_ankle_hip_offset = right_side_metrics["Knee Flexion angle"], right_side_metrics["Tibial angles"], right_side_metrics["Ankle Hip Horizontal Offset"]
    left_trunk_angles, left_ankle_velocities, left_ankle_approach_angles = left_side_metrics["Trunk Angles"], left_side_metrics["Ankle Velocities"], left_side_metrics["Ankle Approach angles"]
    right_trunk_angles, right_ankle_velocities, right_ankle_approach_angles = right_side_metrics["Trunk Angles"], right_side_metrics["Ankle Velocities"], right_side_metrics["Ankle Approach angles"]
    left_hip_osc = calculate_avg_hip_oscillations(left_ankle_peaks, left_hip_coords_smooth)
    right_hip_osc = calculate_avg_hip_oscillations(right_ankle_peaks, right_hip_coords_smooth)
    avg_osc = (left_hip_osc + right_hip_osc)/2
    plt.plot(frames, right_hip_coords_smooth)

    return {
            "Cadence": cadence,
            "Average VO": avg_osc,
            "Left Knee Flexion": left_knee_flexion,
            "Right Knee Flexion": right_knee_flexion,
            "Left Tibial Angles": left_tibial_angles,
            "Right Tibial Angles": right_tibial_angles,
            "Left Ankle Hip Offset": left_ankle_hip_offset,
            "Right Ankle Hip Offset": right_ankle_hip_offset,
            "Left Trunk Angles": left_trunk_angles,
            "Right Trunk Angles": right_trunk_angles,
            "Left Ankle Velocities": left_ankle_velocities,
            "Right Ankle Velocities": right_ankle_velocities,
            "Left Ankle Approach Angles": left_ankle_approach_angles,
            "Right Ankle Approach Angles": right_ankle_approach_angles,
            "Left Strike Frames": left_ankle_peaks,
            "Right Strike Frames": right_ankle_peaks
        }

metrics()
#Comment